# Uber NCR Booking Case Study

In [1]:
import gdown
from google.cloud import storage
import apache_beam as beam

print("All installed ")

All installed 


In [3]:
from airflow import DAG
from airflow.providers.google.cloud.operators.dataflow import DataflowPythonOperator
from datetime import datetime

AirflowOptionalProviderFeatureException: Failed to import apache-airflow-providers-apache-beam. To use the Dataflow service with Apache Beam pipelines, please install the apache-beam provider: pip install apache-airflow-providers-apache-beam

In [4]:
import os
import tempfile
from pathlib import Path
import gdown
from google.cloud import storage 
from datetime import datetime
import pandas as pd
from io import BytesIO
import apache_beam as beam
#from apache_beam.options.pipeline_options import PipelineOptions
import re

from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  

False

## Extract Raw from Drive and upload to Bucket

In [5]:
print("Extracting uber raw data (csv) from google drive")

# - Download CSV content from Google Drive into a temporary file path 
with tempfile.NamedTemporaryFile(delete=False, suffix=".csv") as tmp:
    temp_path = tmp.name

file_id = "1GRyRW2pW3isbbwBPW7lcJrS-yeuZGPj_"
gdown.download(f"https://drive.google.com/uc?id={file_id}", temp_path, quiet=False)
uber_df = pd.read_csv(temp_path)


Extracting uber raw data (csv) from google drive


Downloading...
From: https://drive.google.com/uc?id=1GRyRW2pW3isbbwBPW7lcJrS-yeuZGPj_
To: /var/folders/s6/7k4fm17x4tddpyq7pqjhgp6h0000gn/T/tmp8tnbdwxm.csv
100%|██████████| 25.5M/25.5M [00:00<00:00, 37.7MB/s]


In [3]:
uber_df.head()

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,NaN,NaN,NaN,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,NaN,NaN,NaN,NaN,NaN,416.0,34.02,4.6,5.0,UPI
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,NaN,NaN,NaN,NaN,NaN,737.0,48.21,4.1,4.3,UPI


In [4]:
uber_df.shape

(150000, 21)

In [7]:
# Upload raw to google bucket
PROJECT_ID   = os.getenv("GCP_PROJECT_ID", "dedwhetl")
BUCKET_NAME  = os.getenv("GCS_BUCKET")       
BUCKET_NAME2  = os.getenv("GCS_BUCKET2")        
OBJECT_PATH  = os.getenv("GCS_OBJECT", "trips/ncr_ride_bookings.csv")
GOOGLE_APPLICATION_CREDENTIALS = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
OUT_PREFIX   = os.getenv("OUT_PREFIX")    


if not BUCKET_NAME or not GOOGLE_APPLICATION_CREDENTIALS:
    raise SystemExit("Set GCS_BUCKET and GOOGLE_APPLICATION_CREDENTIALS in .env")


In [8]:
# Convert df → CSV bytes (no temp file)
uber_csv = uber_df.to_csv(index=False).encode("utf-8")


# --- Upload to GCS ---
client = storage.Client() 
bucket = client.bucket(BUCKET_NAME)
blob = bucket.blob(OBJECT_PATH)
blob.upload_from_filename(temp_path, content_type="text/csv")

print(f"Uploaded DataFrame to gs://{BUCKET_NAME}/{OBJECT_PATH}")

os.remove(temp_path)


Uploaded DataFrame to gs://uber-raw/trips/ncr_ride_bookings.csv


## Cleaning + Transformation Steps

1. Ingestion
      > Load raw CSV from GCS bucket into DataFrame.

2. Cleaning
      >- Standardize column names (snake_case).
      >- Parse date + time into booking_timestamp.
      >- Build date_id as YYYYMMDD.
      >- Handle Missing Values
      >>- Columns like Avg VTAT, Avg CTAT, Booking Value, Ride Distance, Driver Ratings, Customer Rating have partial availability.
      >>- don’t drop → keep nulls (fact tables should store reality).
      >>- Cancellation/incomplete columns are sparse (only when that event occurred)
      >- Strip/normalize string columns (status, payment method, vehicle_type).
      >- Ensure IDs (booking_id, customer_id) are clean strings.Remove quotes ("CID12345" → CID12345).

3. Dimension Building
      >- Deduplicate values per domain.
      >- Assign surrogate keys (factorize or enumerate).
      >- dim_date, dim_customer, dim_vehicle, dim_location, dim_payment, dim_reason.

4. Fact Table Assembly
      >- Replace text attributes with surrogate IDs (lookup).
      >- Keep numeric and status metrics in fact (Metrics: Avg VTAT, Avg CTAT, Booking Value, Ride Distance, Driver Ratings, Customer Rating.)
      >- fact_booking table.

5. Load Back to GCS
      > Write all dims + fact as CSV into transformed/ folder in the bucket.

In [9]:
# Standardize column names (snake_case)
uber_df.columns = (
        uber_df.columns.str.strip()
                  .str.replace(r"[^\w\s]", "", regex=True)
                  .str.replace(r"\s+", "_", regex=True)
                  .str.lower()
    )

In [10]:
# Expected columns (best-effort based on your file)
    # date, time, booking_id, booking_status, customer_id, vehicle_type,
    # pickup_location, drop_location, avg_vtat, avg_ctat, booking_value,
    # ride_distance, driver_ratings, customer_rating, payment_method,
    # reason_for_cancelling, incomplete_rides_reason


# Combine date + time → timestamp; build date_id (YYYYMMDD int)
if {"date", "time"}.issubset(uber_df.columns):
        ts = pd.to_datetime(uber_df["date"].astype(str) + " " + uber_df["time"].astype(str), errors="coerce")
elif "date" in uber_df.columns:
        ts = pd.to_datetime(uber_df["date"], errors="coerce")
else:
        ts = pd.NaT

uber_df["booking_timestamp"] = ts
uber_df["booking_date"] = uber_df["booking_timestamp"].dt.date
uber_df["date_id"] = uber_df["booking_timestamp"].dt.strftime("%Y%m%d").astype("Int64")
uber_df.head()

,date,time,booking_id,booking_status,customer_id,vehicle_type,pickup_location,drop_location,avg_vtat,avg_ctat,...,incomplete_rides,incomplete_rides_reason,booking_value,ride_distance,driver_ratings,customer_rating,payment_method,booking_timestamp,booking_date,date_id
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-03-23 12:29:38,2024-03-23,20240323
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI,2024-11-29 18:01:39,2024-11-29,20241129
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card,2024-08-23 08:56:10,2024-08-23,20240823
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,NaN,NaN,416.0,34.02,4.6,5.0,UPI,2024-10-21 17:17:25,2024-10-21,20241021
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,NaN,NaN,737.0,48.21,4.1,4.3,UPI,2024-09-16 22:08:00,2024-09-16,20240916


In [11]:
# IDs as strings (remove stray quotes)
for c in ["booking_id", "customer_id"]:
        if c in uber_df.columns:
            uber_df[c] = uber_df[c].astype(str).str.replace(r'^"+|"+$', "", regex=True)
uber_df.head()

,date,time,booking_id,booking_status,customer_id,vehicle_type,pickup_location,drop_location,avg_vtat,avg_ctat,...,incomplete_rides,incomplete_rides_reason,booking_value,ride_distance,driver_ratings,customer_rating,payment_method,booking_timestamp,booking_date,date_id
0,2024-03-23,12:29:38,CNR5884300,No Driver Found,CID1982111,eBike,Palam Vihar,Jhilmil,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-03-23 12:29:38,2024-03-23,20240323
1,2024-11-29,18:01:39,CNR1326809,Incomplete,CID4604802,Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI,2024-11-29 18:01:39,2024-11-29,20241129
2,2024-08-23,08:56:10,CNR8494506,Completed,CID9202816,Auto,Khandsa,Malviya Nagar,13.4,25.8,...,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card,2024-08-23 08:56:10,2024-08-23,20240823
3,2024-10-21,17:17:25,CNR8906825,Completed,CID2610914,Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,NaN,NaN,416.0,34.02,4.6,5.0,UPI,2024-10-21 17:17:25,2024-10-21,20241021
4,2024-09-16,22:08:00,CNR1950162,Completed,CID9933542,Bike,Ghitorni Village,Khan Market,5.3,19.6,...,NaN,NaN,737.0,48.21,4.1,4.3,UPI,2024-09-16 22:08:00,2024-09-16,20240916


In [12]:
 # Normalize categoricals
if "booking_status" in uber_df.columns:
       uber_df["booking_status"] = uber_df["booking_status"].astype(str).str.strip().str.title()

if "payment_method" in uber_df.columns:
        pm_map = {
            "Upi": "UPI",
            "Debit Card": "Debit Card",
            "Credit Card": "Credit Card",
            "Cash": "Cash",
        }
        uber_df["payment_method"] = (
            uber_df["payment_method"].astype(str).str.strip().replace(pm_map)
        )

for c in ["vehicle_type", "pickup_location", "drop_location"]:
        if c in uber_df.columns:
            uber_df[c] = uber_df[c].astype(str).str.strip()
uber_df.head()

,date,time,booking_id,booking_status,customer_id,vehicle_type,pickup_location,drop_location,avg_vtat,avg_ctat,...,incomplete_rides,incomplete_rides_reason,booking_value,ride_distance,driver_ratings,customer_rating,payment_method,booking_timestamp,booking_date,date_id
0,2024-03-23,12:29:38,CNR5884300,No Driver Found,CID1982111,eBike,Palam Vihar,Jhilmil,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,nan,2024-03-23 12:29:38,2024-03-23,20240323
1,2024-11-29,18:01:39,CNR1326809,Incomplete,CID4604802,Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI,2024-11-29 18:01:39,2024-11-29,20241129
2,2024-08-23,08:56:10,CNR8494506,Completed,CID9202816,Auto,Khandsa,Malviya Nagar,13.4,25.8,...,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card,2024-08-23 08:56:10,2024-08-23,20240823
3,2024-10-21,17:17:25,CNR8906825,Completed,CID2610914,Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,NaN,NaN,416.0,34.02,4.6,5.0,UPI,2024-10-21 17:17:25,2024-10-21,20241021
4,2024-09-16,22:08:00,CNR1950162,Completed,CID9933542,Bike,Ghitorni Village,Khan Market,5.3,19.6,...,NaN,NaN,737.0,48.21,4.1,4.3,UPI,2024-09-16 22:08:00,2024-09-16,20240916


## 3 Build Dimensions and Fact ----

In [13]:
    # dim_date
dim_date = (
        uber_df.loc[uber_df["booking_date"].notna(), ["date_id", "booking_date"]]
          .drop_duplicates()
          .rename(columns={"booking_date": "date"})
          .sort_values("date_id")
    )
    # add handy attributes
if not dim_date.empty:
        dts = pd.to_datetime(dim_date["date"])
        dim_date["year"] = dts.dt.year
        dim_date["quarter"] = dts.dt.quarter
        dim_date["month"] = dts.dt.month
        dim_date["day_of_week"] = dts.dt.day_name()
dim_date = dim_date.astype({"date_id":"Int64"})
dim_date.head()

,date_id,date,year,quarter,month,day_of_week
993,20240101,2024-01-01,2024,1,1,Monday
399,20240102,2024-01-02,2024,1,1,Tuesday
437,20240103,2024-01-03,2024,1,1,Wednesday
177,20240104,2024-01-04,2024,1,1,Thursday
527,20240105,2024-01-05,2024,1,1,Friday


In [14]:
    # dim_customer
if "customer_id" in uber_df.columns:
        dim_customer = uber_df[["customer_id"]].dropna().drop_duplicates().reset_index(drop=True)
        dim_customer["customer_sk"] = pd.factorize(dim_customer["customer_id"])[0] + 1
else:
        dim_customer = pd.DataFrame(columns=["customer_id","customer_sk"])
dim_customer.head()

,customer_id,customer_sk
0,CID1982111,1
1,CID4604802,2
2,CID9202816,3
3,CID2610914,4
4,CID9933542,5


In [15]:
    # dim_vehicle
if "vehicle_type" in uber_df.columns:
        dim_vehicle = uber_df[["vehicle_type"]].dropna().drop_duplicates().reset_index(drop=True)
        dim_vehicle["vehicle_type_id"] = pd.factorize(dim_vehicle["vehicle_type"])[0] + 1
else:
        dim_vehicle = pd.DataFrame(columns=["vehicle_type", "vehicle_type_id"])
dim_vehicle.head()

,vehicle_type,vehicle_type_id
0,eBike,1
1,Go Sedan,2
2,Auto,3
3,Premier Sedan,4
4,Bike,5


In [16]:
    # dim_location (one table, referenced twice)
loc_vals = pd.Series(dtype="object")
if "pickup_location" in uber_df.columns:
        loc_vals = pd.concat([loc_vals, uber_df["pickup_location"].dropna().astype(str)])
if "drop_location" in uber_df.columns:
        loc_vals = pd.concat([loc_vals, uber_df["drop_location"].dropna().astype(str)])
loc_vals = loc_vals.drop_duplicates().reset_index(drop=True)
dim_location = pd.DataFrame({"location_name": loc_vals})
if not dim_location.empty:
        dim_location["location_id"] = pd.factorize(dim_location["location_name"])[0] + 1
else:
        dim_location = pd.DataFrame(columns=["location_name", "location_id"])
dim_location.head()

,location_name,location_id
0,Palam Vihar,1
1,Shastri Nagar,2
2,Khandsa,3
3,Central Secretariat,4
4,Ghitorni Village,5


In [18]:
    # dim_payment
if "payment_method" in uber_df.columns:
        dim_payment = uber_df[["payment_method"]].dropna().drop_duplicates().reset_index(drop=True)
        dim_payment["payment_id"] = pd.factorize(dim_payment["payment_method"])[0] + 1
else:
        dim_payment = pd.DataFrame(columns=["payment_method", "payment_id"])
dim_payment.head()

,payment_method,payment_id
0,nan,1
1,UPI,2
2,Debit Card,3
3,Cash,4
4,Uber Wallet,5


In [19]:
# dim_reason (union of cancellation + incomplete reasons)
reasons = []

if "reason_for_cancelling_by_customer" in uber_df.columns:
        reasons.append(uber_df["reason_for_cancelling_by_customer"].dropna().astype(str))
if "driver_cancellation_reason" in uber_df.columns:
        reasons.append(uber_df["driver_cancellation_reason"].dropna().astype(str))
if "incomplete_rides_reason" in uber_df.columns:
        reasons.append(uber_df["incomplete_rides_reason"].dropna().astype(str))
if reasons:
        reasons = pd.concat(reasons).drop_duplicates().reset_index(drop=True)
else:
        reasons = pd.Series(dtype="object")
dim_reason = pd.DataFrame({"reason_text": reasons})
if not dim_reason.empty:
        dim_reason["reason_id"] = pd.factorize(dim_reason["reason_text"])[0] + 1
else:
        dim_reason = pd.DataFrame(columns=["reason_id", "type"])
dim_reason

,reason_text,reason_id
0,Driver is not moving towards pickup location,1
1,Driver asked to cancel,2
2,AC is not working,3
3,Change of plans,4
4,Wrong Address,5
5,Personal & Car related issues,6
6,Customer related issue,7
7,More than permitted people in there,8
8,The customer was coughing/sick,9
9,Vehicle Breakdown,10


In [22]:
# ---- 5) Fact tables ----

# Build lookup maps from dimension tables
customer_map  = dim_customer.set_index('customer_id')['customer_sk']
vehicle_map  = dim_vehicle.set_index('vehicle_type')['vehicle_type_id']
location_map = dim_location.set_index('location_name')['location_id']
payment_map  = dim_payment.set_index('payment_method')['payment_id']
reason_map   = dim_reason.set_index('reason_text')['reason_id']

fact = uber_df.copy()

# Map to IDs
fact['customer_sk']      = fact['customer_id'].map(customer_map)
fact['vehicle_type_id']    = fact['vehicle_type'].map(vehicle_map)
fact['pickup_location_id'] = fact['pickup_location'].map(location_map)
fact['drop_location_id']   = fact['drop_location'].map(location_map)
fact['payment_id']         = fact['payment_method'].map(payment_map)

# Map multiple reason columns to separate FK IDs
for src_col, fk_col in [
    ('reason_for_cancelling_by_customer', 'customer_cancel_reason_id'),
    ('driver_cancellation_reason',        'driver_cancel_reason_id'),
    ('incomplete_rides_reason',           'incomplete_reason_id'),
]:
    if src_col in fact:
        fact[fk_col] = (
            fact[src_col].dropna().astype(str).str.strip()
            .replace({'': pd.NA})
            .map(reason_map)
        )

# Select final fact columns (keep natural keys that are already IDs)
fact_booking = fact[[
    'booking_id', 'date_id', 'customer_sk',
    'vehicle_type_id', 'pickup_location_id', 'drop_location_id',
    'payment_id', 'customer_cancel_reason_id', 'driver_cancel_reason_id',
    'incomplete_reason_id',
    'booking_status', 'avg_vtat', 'avg_ctat', 'booking_value',
    'ride_distance', 'driver_ratings', 'customer_rating'
]]

fact_booking

,booking_id,date_id,customer_sk,vehicle_type_id,pickup_location_id,drop_location_id,payment_id,customer_cancel_reason_id,driver_cancel_reason_id,incomplete_reason_id,booking_status,avg_vtat,avg_ctat,booking_value,ride_distance,driver_ratings,customer_rating
0,CNR5884300,20240323,1,1,1,76,1,NaN,NaN,NaN,No Driver Found,NaN,NaN,NaN,NaN,NaN,NaN
1,CNR1326809,20241129,2,2,2,105,2,NaN,NaN,10.0,Incomplete,4.9,14.0,237.0,5.73,NaN,NaN
2,CNR8494506,20240823,3,3,3,14,3,NaN,NaN,NaN,Completed,13.4,25.8,627.0,13.58,4.9,4.9
3,CNR8906825,20241021,4,4,4,158,2,NaN,NaN,NaN,Completed,13.1,28.5,416.0,34.02,4.6,5.0
4,CNR1950162,20240916,5,5,5,87,2,NaN,NaN,NaN,Completed,5.3,19.6,737.0,48.21,4.1,4.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149995,CNR6500631,20241111,148784,6,48,61,5,NaN,NaN,NaN,Completed,10.2,44.4,475.0,40.08,3.7,4.1
149996,CNR2468611,20241124,148785,6,118,72,2,NaN,NaN,NaN,Completed,5.1,30.8,1093.0,21.31,4.8,5.0
149997,CNR6358306,20240918,148786,2,69,108,4,NaN,NaN,NaN,Completed,2.7,23.4,852.0,15.93,3.9,4.4
149998,CNR3030099,20241005,148787,3,119,91,2,NaN,NaN,NaN,Completed,6.9,39.6,333.0,45.54,4.1,3.7


## Write out CSVs to GCS

In [27]:

outs = {
        "dim_date.csv": dim_date,
        "dim_customer.csv": dim_customer,
        "dim_vehicle.csv": dim_vehicle,
        "dim_location.csv": dim_location,
        "dim_payment.csv": dim_payment,
        "dim_reason.csv": dim_reason,
        "fact_booking.csv": fact_booking
    }

for object_name, frame in outs.items():
    if frame is None or (hasattr(frame, "empty") and frame.empty):
        print(f"⏭️  Skipping empty output: {object_name}")
        continue

    with tempfile.TemporaryDirectory() as tmpdir:
        bucket = client.bucket(BUCKET_NAME2)  
        blob_path = f"{OUT_PREFIX.rstrip('/')}/{object_name}" if OUT_PREFIX else object_name

        local_path = Path(tmpdir) / Path(object_name).name
        frame.to_csv(local_path, index=False)
        bucket.blob(blob_path).upload_from_filename(str(local_path), content_type="text/csv")
        print(f"Wrote {len(frame):,} rows → gs://{BUCKET_NAME2}/{blob_path}")

print(" Done.")

Wrote 365 rows → gs://uber-transformed/dim/dim_date.csv
Wrote 148,788 rows → gs://uber-transformed/dim/dim_customer.csv
Wrote 7 rows → gs://uber-transformed/dim/dim_vehicle.csv
Wrote 176 rows → gs://uber-transformed/dim/dim_location.csv
Wrote 6 rows → gs://uber-transformed/dim/dim_payment.csv
Wrote 12 rows → gs://uber-transformed/dim/dim_reason.csv
Wrote 150,000 rows → gs://uber-transformed/dim/fact_booking.csv
 Done.
